# Notebook 03 — Modélisation : prédiction de la publication des résultats d'essais cliniques

**Projet** : IA sur données CTIS (Clinical Trials Information System — EMA)  
**Objectif** : Prédire si un essai clinique publiera ses résultats (`Trial results` = Yes/No)  
**Approche** : Classification supervisée binaire  
**Pipeline** : Feature engineering → Preprocessing → Modélisation → Évaluation → Interprétabilité (SHAP)  

---
> **Note académique** : Les corrélations de Pearson observées dans l'EDA (r < 0.35) sont typiques des données de santé réelles à signal diffus. La combinaison non-linéaire des features via XGBoost est justifiée par cette caractéristique.

## 0. Imports et configuration

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Sklearn — preprocessing & pipeline
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Sklearn — modèles
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# XGBoost
from xgboost import XGBClassifier

# Sklearn — métriques
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    f1_score,
    ConfusionMatrixDisplay
)

# SHAP pour l'interprétabilité
import shap

# Reproductibilité
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Imports OK")

Imports OK


## 1. Chargement et préparation de la variable cible

> **Choix de la variable cible** : `Trial results` (Yes/No) est retenu pour sa lisibilité réglementaire.  
> Elle est binarisée : `Yes` → 1 (résultats publiés), `No` → 0 (absence de publication).

In [3]:
df = pd.read_csv("CTIS_clean_data_last.csv")

# Binarisation de la cible
df['target'] = (df['Trial results'].str.strip().str.lower() == 'yes').astype(int)

# Distribution de la cible
print("Distribution de la variable cible :")
print(df['target'].value_counts())
print(f"\nTaux de publication : {df['target'].mean():.2%}")

# Visualisation du déséquilibre
fig, ax = plt.subplots(figsize=(5, 3))
df['target'].value_counts().plot(kind='bar', ax=ax, color=['#5DCAA5', '#D85A30'])
ax.set_xticklabels(['Non publié (0)', 'Publié (1)'], rotation=0)
ax.set_title('Déséquilibre de la variable cible')
ax.set_ylabel('Nombre d\'essais')
plt.tight_layout()
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'CTIS_clean_data_last.csv'

## 2. Feature engineering

> **Principe** : On construit des features à partir des colonnes brutes CTIS.  
> Trois types de features sont retenus : numériques dérivées des dates, catégorielles directes, et une proxy textuelle (longueur du titre).  
> **Vigilance data leakage** : `Last updated`, `Global end of the trial` et `Trial results` sont exclus car postérieurs à l'événement cible.

In [ ]:
# Conversion des dates si pas déjà fait
for col in ['Decision date', 'Start date', 'End date', 'Last updated']:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# --- Features numériques ---

# Durée de l'essai en jours (proxy de complexité)
df['num_trial_duration_days'] = (
    df['End date'] - df['Start date']
).dt.days

# Ancienneté depuis la décision (proxy d'avancement)
reference_date = pd.Timestamp('2024-01-01')
df['num_update_recency_days'] = (
    reference_date - df['Last updated']
).dt.days

# Longueur du titre (proxy de spécificité du protocole)
df['num_title_length'] = df['Title of the trial'].fillna('').str.len()

# Nombre de pays impliqués (proxy de complexité internationale)
df['num_countries_count'] = df['countries'].apply(
    lambda x: len(str(x).split(',')) if pd.notna(x) else 1
)

# --- Features catégorielles ---
cat_features = [
    'Trial phase',
    'Sponsor type',
    'Gender',
    'Age group',
    'Therapeutic area',
    'Trial region',
]

num_features = [
    'num_trial_duration_days',
    'num_update_recency_days',
    'num_title_length',
    'num_countries_count',
]

all_features = num_features + cat_features
TARGET = 'target'

# Sous-ensemble sans NAs sur la cible
df_model = df[all_features + [TARGET]].dropna(subset=[TARGET])

print(f"Dimensions du dataset de modélisation : {df_model.shape}")
print(f"Features numériques : {num_features}")
print(f"Features catégorielles : {cat_features}")

## 3. Split train / test stratifié

> Le split est **stratifié** sur la cible pour conserver le ratio de déséquilibre dans les deux ensembles.  
> Ratio 80/20, conforme aux standards de la littérature ML en santé.

In [ ]:
X = df_model[all_features]
y = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"Taille train : {X_train.shape[0]} observations")
print(f"Taille test  : {X_test.shape[0]} observations")
print(f"Taux positif train : {y_train.mean():.2%}")
print(f"Taux positif test  : {y_test.mean():.2%}")

## 4. Construction du pipeline sklearn

> **Architecture du pipeline** :  
> - Variables numériques : imputation par médiane + standardisation (StandardScaler)  
> - Variables catégorielles : imputation par 'missing' + OneHotEncoder avec `drop='first'`  
>   (le `drop='first'` est **obligatoire** pour éviter la multicolinéarité parfaite détectée dans la heatmap)  
> 
> Le ColumnTransformer applique ces deux traitements en parallèle avant chaque modèle.

In [ ]:
# Préprocesseur numérique
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Préprocesseur catégoriel — drop='first' corrige la multicolinéarité one-hot
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
])

# Assemblage
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, num_features),
    ('cat', categorical_transformer, cat_features)
])

print("Pipeline de préprocessing construit.")

## 5. Définition des 3 modèles

> **Stratégie de comparaison** :  
> 1. **Logistic Regression** : baseline interprétable, coefficients directement lisibles  
> 2. **Random Forest** : ensemble d'arbres, robuste au bruit et à la multicolinéarité  
> 3. **XGBoost** : gradient boosting, généralement le plus performant sur données tabulaires  
> 
> Le paramètre `class_weight='balanced'` (LR, RF) et `scale_pos_weight` (XGBoost) compensent le déséquilibre de classes.

In [ ]:
# Calcul du ratio pour XGBoost
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos = neg / pos
print(f"scale_pos_weight pour XGBoost : {scale_pos:.2f}")

# Définition des pipelines complets
models = {
    'Logistic Regression': Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(
            class_weight='balanced',
            max_iter=1000,
            random_state=RANDOM_STATE
        ))
    ]),
    'Random Forest': Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(
            n_estimators=300,
            class_weight='balanced',
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ]),
    'XGBoost': Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', XGBClassifier(
            n_estimators=300,
            scale_pos_weight=scale_pos,
            eval_metric='logloss',
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbosity=0
        ))
    ]),
}

print("3 pipelines modèles définis.")

## 6. Validation croisée stratifiée (5-fold)

> La **validation croisée stratifiée** (StratifiedKFold, k=5) est la procédure d'évaluation recommandée  
> pour les datasets déséquilibrés. Elle garantit que chaque fold conserve la distribution de la cible.  
> Métriques retenues : **AUC-ROC** (robuste au déséquilibre) et **F1-score** (équilibre précision/rappel).

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

results_cv = {}

for name, pipeline in models.items():
    print(f"Évaluation : {name}...")
    auc_scores = cross_val_score(
        pipeline, X_train, y_train,
        cv=cv, scoring='roc_auc', n_jobs=-1
    )
    f1_scores = cross_val_score(
        pipeline, X_train, y_train,
        cv=cv, scoring='f1', n_jobs=-1
    )
    results_cv[name] = {
        'AUC mean': auc_scores.mean(),
        'AUC std': auc_scores.std(),
        'F1 mean': f1_scores.mean(),
        'F1 std': f1_scores.std()
    }
    print(f"  AUC = {auc_scores.mean():.3f} ± {auc_scores.std():.3f}")
    print(f"  F1  = {f1_scores.mean():.3f} ± {f1_scores.std():.3f}")

# Tableau récapitulatif
df_results = pd.DataFrame(results_cv).T.round(3)
print("\n--- Tableau comparatif ---")
print(df_results)

## 7. Entraînement final et courbes ROC comparatives

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

trained_models = {}
colors = ['#534AB7', '#1D9E75', '#D85A30']

# --- Courbes ROC ---
ax_roc = axes[0]
ax_roc.plot([0, 1], [0, 1], 'k--', lw=1, label='Aléatoire (AUC = 0.50)')

for (name, pipeline), color in zip(models.items(), colors):
    pipeline.fit(X_train, y_train)
    trained_models[name] = pipeline
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_prob)
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    ax_roc.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC = {auc:.3f})')

ax_roc.set_xlabel('Taux faux positifs (1 - Spécificité)')
ax_roc.set_ylabel('Taux vrais positifs (Sensibilité)')
ax_roc.set_title('Courbes ROC — comparaison des 3 modèles')
ax_roc.legend(loc='lower right', fontsize=9)
ax_roc.grid(alpha=0.3)

# --- Barplot AUC vs F1 ---
ax_bar = axes[1]
x = np.arange(len(results_cv))
width = 0.35
bars1 = ax_bar.bar(x - width/2,
                    [v['AUC mean'] for v in results_cv.values()],
                    width, label='AUC-ROC (CV)', color='#534AB7', alpha=0.85)
bars2 = ax_bar.bar(x + width/2,
                    [v['F1 mean'] for v in results_cv.values()],
                    width, label='F1-score (CV)', color='#1D9E75', alpha=0.85)
ax_bar.set_xticks(x)
ax_bar.set_xticklabels(results_cv.keys(), rotation=10, ha='right', fontsize=9)
ax_bar.set_ylim(0, 1)
ax_bar.set_title('Métriques de validation croisée (5-fold)')
ax_bar.legend()
ax_bar.grid(axis='y', alpha=0.3)

for bar in bars1:
    ax_bar.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax_bar.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('figures/roc_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure sauvegardée dans figures/roc_comparison.png")

## 8. Matrices de confusion — modèle retenu (XGBoost)

> On analyse en détail le modèle le plus performant.  
> La matrice de confusion permet de lire les **faux négatifs** (essais qui auraient dû publier  
> mais ne l'ont pas fait) — qui sont le risque réglementaire principal dans ce contexte.

In [ ]:
best_model_name = 'XGBoost'
best_model = trained_models[best_model_name]

y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print(f"=== Rapport de classification — {best_model_name} ===")
print(classification_report(y_test, y_pred,
                             target_names=['Non publié (0)', 'Publié (1)']))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['Non publié', 'Publié'],
    colorbar=False,
    ax=ax,
    cmap='Blues'
)
ax.set_title(f'Matrice de confusion — {best_model_name}')
plt.tight_layout()
plt.savefig('figures/confusion_matrix_xgboost.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Interprétabilité — SHAP (SHapley Additive exPlanations)

> **Pourquoi SHAP ?**  
> SHAP est fondé sur la théorie des jeux coopératifs (Shapley, 1953).  
> Il attribue à chaque feature sa contribution marginale à la prédiction, de façon cohérente et additive.  
> Dans notre contexte, SHAP permet de répondre à :  
> *"Pourquoi ce modèle prédit-il qu'un essai ne publiera pas ses résultats ?"*  
> — ce qui constitue une valeur directe pour un comité éthique ou une autorité réglementaire.

In [ ]:
# Extraction du préprocesseur et du classifieur
xgb_preprocessor = best_model.named_steps['preprocessor']
xgb_classifier = best_model.named_steps['classifier']

# Transformation des données test
X_test_transformed = xgb_preprocessor.transform(X_test)

# Récupération des noms de features après encodage
num_feature_names = num_features
cat_feature_names = list(
    xgb_preprocessor
    .named_transformers_['cat']
    .named_steps['onehot']
    .get_feature_names_out(cat_features)
)
all_feature_names = num_feature_names + cat_feature_names

# DataFrame transformé avec noms
X_test_df = pd.DataFrame(X_test_transformed, columns=all_feature_names)

# Calcul des valeurs SHAP
explainer = shap.TreeExplainer(xgb_classifier)
shap_values = explainer.shap_values(X_test_df)

print("Valeurs SHAP calculées.")
print(f"Forme : {shap_values.shape}")

In [ ]:
# --- SHAP Summary Plot (vue globale) ---
plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_values,
    X_test_df,
    plot_type='dot',
    max_display=15,
    show=False
)
plt.title('SHAP Summary Plot — importance et direction des features\n(rouge = valeur élevée, bleu = valeur faible)', fontsize=11)
plt.tight_layout()
plt.savefig('figures/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- SHAP Bar Plot (importance absolue) ---
plt.figure(figsize=(8, 5))
shap.summary_plot(
    shap_values,
    X_test_df,
    plot_type='bar',
    max_display=12,
    show=False
)
plt.title('SHAP — importance absolue moyenne des features', fontsize=11)
plt.tight_layout()
plt.savefig('figures/shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- SHAP Waterfall — explication d'une prédiction individuelle ---
# On prend le premier essai de l'ensemble test comme exemple pédagogique
idx_example = 0

shap_explanation = shap.Explanation(
    values=shap_values[idx_example],
    base_values=explainer.expected_value,
    data=X_test_df.iloc[idx_example],
    feature_names=all_feature_names
)

plt.figure(figsize=(10, 6))
shap.waterfall_plot(shap_explanation, max_display=12, show=False)
plt.title(f'SHAP Waterfall — explication individuelle (essai #{idx_example})', fontsize=11)
plt.tight_layout()
plt.savefig('figures/shap_waterfall_example.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Prédiction pour cet essai : {y_pred[idx_example]} (réalité : {y_test.iloc[idx_example]})")

## 10. Sauvegarde du modèle retenu

> Le modèle final est sérialisé avec `joblib` pour pouvoir être rechargé sans réentraînement.  
> Le pipeline complet (préprocesseur + classifieur) est sauvegardé d'un bloc.

In [ ]:
import joblib
import os

os.makedirs('models', exist_ok=True)
os.makedirs('figures', exist_ok=True)

model_path = 'models/xgboost_ctis_pipeline.pkl'
joblib.dump(best_model, model_path)
print(f"Modèle sauvegardé : {model_path}")

# Vérification du rechargement
model_reloaded = joblib.load(model_path)
y_check = model_reloaded.predict(X_test[:5])
print(f"Vérification rechargement : {y_check}")

## 11. Synthèse des résultats

> Cette cellule génère un tableau de synthèse pour intégration dans un rapport ou README.

In [ ]:
print("=" * 55)
print("SYNTHÈSE — Modélisation CTIS (Piste A)")
print("=" * 55)
print(f"Tâche         : Classification binaire (Trial results)")
print(f"Dataset       : {df_model.shape[0]} essais, {len(all_features)} features")
print(f"Déséquilibre  : {y.mean():.1%} positifs")
print(f"Évaluation    : StratifiedKFold (k=5)")
print()
print(df_results.to_string())
print()
best_auc = max(results_cv, key=lambda k: results_cv[k]['AUC mean'])
print(f"Meilleur modèle (AUC) : {best_auc}")
print(f"AUC           : {results_cv[best_auc]['AUC mean']:.3f} ± {results_cv[best_auc]['AUC std']:.3f}")
print(f"F1-score      : {results_cv[best_auc]['F1 mean']:.3f} ± {results_cv[best_auc]['F1 std']:.3f}")
print("=" * 55)